# M59i — full KITTI validation bundle
**Revision: M59i-2026-09-19-r1. Run all three code cells top to bottom. CPU runtime is enough.**

This packages all **3,769 Chen validation images**, original calibration and original KITTI labels from your existing Drive dataset. No training, inference, model export, upstream clone, CUDA build or GPU is needed. Core ML and PyTorch comparison runs later on the Mac.

Required: canonical `kitti_chen/val.txt`, the original M58 `m58_reference_io.npz`, and KITTI images/calibration/**unmodified labels**. Do not use the relabeled MonoDGP training labels. The script verifies the original label hash and the 16 reviewed image/calibration hashes.

Return the **single ZIP** printed by cell 3. This contains dataset files and may be several GB; ensure enough Drive space for the copied subset plus its ZIP. Restart: rerun these same three cells; identical copied files are reused. If files differ, it stops without deleting them. No full validation result is claimed by this notebook.

In [ ]:
# M59i-2026-09-19-r1 — all imports, paths and helpers are defined here.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, subprocess, sys
REVISION = 'M59i-2026-09-19-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
MYDRIVE = Path('/content/drive/MyDrive')
# Change this one path if your ORIGINAL KITTI dataset is elsewhere.
DRIVE_KITTI = MYDRIVE/'datasets/kitti'
SPLIT_FILE = MYDRIVE/'mobile_adas3d_splits/kitti_chen/val.txt'
ANCHOR = MYDRIVE/'mobile_adas3d_outputs/compression/monodgp_m58_coreml_conversion/m58_reference_io.npz'
OUTPUT_ROOT = MYDRIVE/'mobile_adas3d_outputs/compression/monodgp_m59i_full_validation'
BUNDLE = OUTPUT_ROOT/'m59i_chen_val3769'
LOGS = OUTPUT_ROOT/'colab_logs'
LOGS.mkdir(parents=True, exist_ok=True)
def run_logged(command, log_path, cwd=None):
    command = [str(x) for x in command]
    print('+', shlex.join(command), flush=True)
    tail = deque(maxlen=40)
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        code = process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
print(REVISION, 'CPU data packaging only.')


In [ ]:
# Get current main without deleting/resetting local changes.
REPO_URL = '/'.join(('https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'))
if not MOBILE_REPO.exists():
    run_logged(['git','clone','--branch','main',REPO_URL,MOBILE_REPO], LOGS/'clone.log')
if not (MOBILE_REPO/'.git').exists(): raise RuntimeError(f'Not a Git checkout: {MOBILE_REPO}')
branch = subprocess.check_output(['git','branch','--show-current'], cwd=MOBILE_REPO, text=True).strip()
dirty = subprocess.check_output(['git','status','--porcelain'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main' or dirty:
    raise RuntimeError(f'Expected clean main. Preserve your edits before syncing; branch={branch}\n{dirty}')
run_logged(['git','pull','--ff-only'], LOGS/'sync.log', MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/collect_monodgp_m59i_validation.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'Current main must contain {SCRIPT}')
run_logged([sys.executable,'-m','pip','install','-q','numpy>=2.0,<2.4','Pillow','opencv-python-headless'], LOGS/'dependencies.log')
print('Project commit:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=MOBILE_REPO, text=True).strip())
print(REVISION, 'setup complete')


In [ ]:
# Package the exact validation subset and original GT; resumable after interruption.
for path in (SPLIT_FILE, ANCHOR):
    if not path.is_file(): raise FileNotFoundError(f'Required artifact missing: {path}')
sys.path.insert(0, str(MOBILE_REPO))
from scripts.collect_monodgp_m59i_validation import full_ids, label_hash, LABEL_TREE_SHA256
IDS = full_ids(SPLIT_FILE)
roots = [DRIVE_KITTI, Path('/content/kitti'), Path('/content/monodgp_kitti_m56d')]
def find_directory(names, suffix, original_labels=False):
    candidates = [root/'training'/name for root in roots for name in names]
    for folder in candidates:
        if all((folder/(sample_id+suffix)).is_file() for sample_id in IDS):
            if original_labels and label_hash(folder, IDS) != LABEL_TREE_SHA256:
                print('Skipping remapped/changed labels:', folder); continue
            return folder
    raise FileNotFoundError('Cannot locate all original validation files. Set DRIVE_KITTI in cell 1. Tried: '+', '.join(map(str,candidates)))
IMAGE_DIR = find_directory(('image_2','image_02'), '.png')
CALIB_DIR = find_directory(('calib',), '.txt')
LABEL_DIR = find_directory(('label_2','label_02'), '.txt', original_labels=True)
print('Images:', IMAGE_DIR, '\nCalibration:', CALIB_DIR, '\nOriginal labels:', LABEL_DIR)
run_logged([sys.executable,'-u',SCRIPT,'--image-dir',IMAGE_DIR,'--calibration-dir',CALIB_DIR,
            '--label-dir',LABEL_DIR,'--split-file',SPLIT_FILE,'--anchor-npz',ANCHOR,'--output-dir',BUNDLE],
           LOGS/'m59i_collect.log', MOBILE_REPO)
manifest = json.loads((BUNDLE/'m59i_dataset_manifest.json').read_text())
assert manifest['complete'] and manifest['sample_count'] == 3769
ZIP = BUNDLE.parent/(BUNDLE.name+'.zip')
print('DONE. Download and return this ZIP:', ZIP)
print('Size (GB):', round(ZIP.stat().st_size/1e9, 2))
print('Next: paired PyTorch/Core ML full validation on the Mac. No training or GPU is needed here.')
